# 10 — Benchmark final de algoritmos clásicos

## Proyecto
**Kaki_DeepHS_StorageDays**

## Objetivo

Cerrar la fase principal de machine learning clásico con una comparación compacta y rigurosa de varios algoritmos usando exactamente los mismos splits fruit-disjoint ya congelados.

Se evalúan dos tareas principales sobre **VIS_COR**, el sensor que ha mostrado el mejor desempeño:

### Tarea A — Single-scan / espectro absoluto

\[
R_t \rightarrow \text{storage days}
\]

- 199 fruit-days
- 40 frutos
- 249 bandas VIS_COR
- incluye día 0
- representa la tarea de estimación a partir de una sola medición

### Tarea B — Longitudinal / baseline-referenced

\[
SNV(R_t)-SNV(R_0)
\rightarrow \text{storage days}
\]

- 159 fruit-days
- 38 frutos con observaciones posteriores al día 0
- días 1,2,3,4,6,7,8
- requiere el baseline del mismo fruto

## Algoritmos

Se comparan cinco algoritmos:

1. **Ridge Regression**
2. **PLS Regression**
3. **SVR con kernel RBF**
4. **Random Forest Regression**
5. **Extra Trees Regression**

## Métricas

Se reportan al menos ocho métricas:

- MAE
- RMSE
- \(R^2\)
- Spearman \(\rho\)
- accuracy dentro de ±1 día
- MAE fruit-balanced
- MAE day-balanced
- bias

La métrica primaria para selección y comparación es:

\[
\boxed{\text{MAE fruit-balanced}}
\]

## Validación

- outer 5-fold fruit-disjoint congelado
- inner 4-fold fruit-disjoint congelado
- tuning únicamente dentro de development
- test externo usado una sola vez
- bootstrap pareado por fruto
- análisis por día
- stress test `session-excluded` con null correcto

## Interpretación

Este notebook no busca "ganar" con el algoritmo más complejo. Busca comprobar si las conclusiones principales dependen de Ridge o si son robustas a familias de modelos lineales y no lineales.


### Public reproducibility version

This notebook is the cleaned repository version used to reproduce the manuscript analyses. The scientific workflow, frozen fruit-disjoint partitions, model-selection logic, seeds, and evaluation metrics are preserved. Google Drive mounting and chat-specific ZIP export steps were removed. Paths are relative to the repository root. Raw DeepHS Fruit data are not redistributed; see `../docs/DATASET.md`.


In [ ]:
from pathlib import Path
import os

def find_repository_root():
    """Locate the cloned repository or use PERSIMMON_HSI_ROOT if defined."""
    env_root = os.environ.get("PERSIMMON_HSI_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()

    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate

    if cwd.name == "notebooks":
        return cwd.parent

    return cwd

PROJECT_ROOT = find_repository_root()
print("Repository root:", PROJECT_ROOT)


## 1. Librerías, rutas y configuración


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import itertools
import json
import math
import platform
import shutil
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_NAME = "Kaki_DeepHS_StorageDays"
NOTEBOOK_ID = "10_kaki_final_classical_ml_benchmark"
RUN_SIGNATURE = "final_classical_benchmark_v1"

ROOT = PROJECT_ROOT

INTERIM = ROOT / "data/interim"
SPLITS_DIR = ROOT / "data/splits"

RESULTS = ROOT / "results/10_kaki_final_classical_ml_benchmark"
FIGURES = ROOT / "figures/10_kaki_final_classical_ml_benchmark"
TABLES = ROOT / "tables/10_kaki_final_classical_ml_benchmark"
EXPORT_ROOT = ROOT / "exports/10_kaki_final_classical_ml_benchmark"
CHECKPOINT_DIR = RESULTS / f"checkpoints_{RUN_SIGNATURE}"

for p in [
    RESULTS,
    FIGURES,
    TABLES,
    EXPORT_ROOT,
    CHECKPOINT_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

CAMERA = "VIS_COR"

TASKS = [
    "absolute",
    "delta_snv",
]

ALGORITHMS = [
    "Ridge",
    "PLSR",
    "SVR_RBF",
    "RandomForest",
    "ExtraTrees",
]

OUTER_FOLDS = 5
INNER_FOLDS = 4

RANDOM_SEED = 20260818
BOOTSTRAP_REPS = 5000

RESUME = True

print("Camera:", CAMERA)
print("Tasks:", TASKS)
print("Algorithms:", ALGORITHMS)
print("Resume:", RESUME)


## 2. Cargar VIS_COR, QC y splits congelados


In [ ]:
SPECTRA_PATH = (
    INTERIM /
    "kaki_fruit_day_spectra_VIS_COR.csv"
)

CUBE_PATH = (
    INTERIM /
    "kaki_cube_spectra_VIS_COR.csv"
)

OUTER_SPLIT_PATH = (
    SPLITS_DIR /
    "kaki_outer5_fruit_disjoint_v1.csv"
)

INNER_SPLIT_PATH = (
    SPLITS_DIR /
    "kaki_inner4_within_outer5_fruit_disjoint_v1.csv"
)

for p in [
    SPECTRA_PATH,
    CUBE_PATH,
    OUTER_SPLIT_PATH,
    INNER_SPLIT_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(
            f"Falta archivo requerido: {p}"
        )

cube = pd.read_csv(
    CUBE_PATH
)

assert len(cube) == 398
assert float(
    cube[
        "mask_coverage"
    ].median()
) < 0.87
assert (
    cube[
        "mask_qc_flag"
    ]
    ==
    "ok"
).all()
assert cube[
    "extraction_error"
].isna().all()

data = pd.read_csv(
    SPECTRA_PATH
)

data[
    "fruit_id"
] = pd.to_numeric(
    data[
        "fruit_id"
    ],
    errors="raise"
).astype(int)

data[
    "storage_days_model"
] = pd.to_numeric(
    data[
        "storage_days_model"
    ],
    errors="raise"
).astype(int)

wl_cols = [
    c
    for c in data.columns
    if c.startswith(
        "wl_"
    )
]

wavelengths = np.array(
    [
        float(
            c.replace(
                "wl_",
                ""
            )
        )
        for c in wl_cols
    ],
    dtype=float
)

assert len(data) == 199
assert data["fruit_id"].nunique() == 40
assert len(wl_cols) == 249
assert not data[wl_cols].isna().any().any()

data = data.sort_values(
    [
        "fruit_id",
        "storage_days_model"
    ]
).reset_index(
    drop=True
)

outer = pd.read_csv(
    OUTER_SPLIT_PATH
)

inner = pd.read_csv(
    INNER_SPLIT_PATH
)

for df in [
    outer,
    inner,
]:

    df[
        "fruit_id"
    ] = pd.to_numeric(
        df[
            "fruit_id"
        ],
        errors="raise"
    ).astype(int)

outer[
    "outer_fold"
] = pd.to_numeric(
    outer[
        "outer_fold"
    ],
    errors="raise"
).astype(int)

inner[
    "outer_fold"
] = pd.to_numeric(
    inner[
        "outer_fold"
    ],
    errors="raise"
).astype(int)

inner[
    "inner_fold"
] = pd.to_numeric(
    inner[
        "inner_fold"
    ],
    errors="raise"
).astype(int)

print(
    "VIS_COR:",
    len(data),
    "fruit-days |",
    data["fruit_id"].nunique(),
    "frutos |",
    len(wl_cols),
    "bandas"
)


## 3. Auditoría anti-leakage


In [ ]:
all_fruits = set(
    outer[
        "fruit_id"
    ].astype(int)
)

audit_rows = []

for outer_fold in range(
    OUTER_FOLDS
):

    test_fruits = set(
        outer.loc[
            outer[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    dev_fruits = (
        all_fruits
        -
        test_fruits
    )

    inner_fruits = set(
        inner.loc[
            inner[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    audit_rows.append({
        "outer_fold":
            int(
                outer_fold
            ),

        "n_dev_fruits":
            len(
                dev_fruits
            ),

        "n_test_fruits":
            len(
                test_fruits
            ),

        "train_test_overlap":
            len(
                dev_fruits
                &
                test_fruits
            ),

        "inner_test_overlap":
            len(
                inner_fruits
                &
                test_fruits
            ),

        "inner_matches_dev":
            inner_fruits
            ==
            dev_fruits,
    })

split_audit = pd.DataFrame(
    audit_rows
)

display(
    split_audit
)

assert (
    split_audit[
        "train_test_overlap"
    ]
    ==
    0
).all()

assert (
    split_audit[
        "inner_test_overlap"
    ]
    ==
    0
).all()

assert split_audit[
    "inner_matches_dev"
].all()

split_audit.to_csv(
    TABLES /
    "01_split_audit.csv",
    index=False
)

print("✅ Splits válidos.")


# 4. Construir las dos tareas

## Absolute

Usa el espectro VIS_COR completo.

## Delta-SNV

Para cada fruto con seguimiento posterior:

\[
X_{i,t}
=
SNV(R_{i,t})
-
SNV(R_{i,0})
\]

Solo incluye \(t>0\).


In [ ]:
def snv_rows(
    X
):

    X = np.asarray(
        X,
        dtype=float
    )

    mu = X.mean(
        axis=1,
        keepdims=True
    )

    sd = X.std(
        axis=1,
        keepdims=True
    )

    sd[
        (~np.isfinite(sd))
        |
        (sd == 0)
    ] = 1.0

    return (
        X
        -
        mu
    ) / sd


absolute_df = data[
    [
        "fruit_id",
        "storage_days_model",
        "day_code",
        *wl_cols,
    ]
].copy()

delta_parts = []

for fruit_id, g in data.groupby(
    "fruit_id"
):

    g = g.sort_values(
        "storage_days_model"
    )

    baseline = g[
        g[
            "storage_days_model"
        ]
        ==
        0
    ]

    assert len(
        baseline
    ) == 1

    later = g[
        g[
            "storage_days_model"
        ]
        >
        0
    ].copy()

    if len(
        later
    ) == 0:
        continue

    X0 = baseline[
        wl_cols
    ].to_numpy(
        dtype=float
    )

    X = later[
        wl_cols
    ].to_numpy(
        dtype=float
    )

    delta_snv = (
        snv_rows(
            X
        )
        -
        snv_rows(
            X0
        )
    )

    out_part = later[
        [
            "fruit_id",
            "storage_days_model",
            "day_code",
        ]
    ].reset_index(
        drop=True
    )

    for j, col in enumerate(
        wl_cols
    ):

        out_part[
            col
        ] = delta_snv[
            :,
            j
        ]

    delta_parts.append(
        out_part
    )

delta_df = pd.concat(
    delta_parts,
    ignore_index=True
)

assert len(absolute_df) == 199
assert absolute_df["fruit_id"].nunique() == 40

assert len(delta_df) == 159
assert delta_df["fruit_id"].nunique() == 38
assert set(
    delta_df[
        "storage_days_model"
    ].unique()
) == {
    1,
    2,
    3,
    4,
    6,
    7,
    8,
}

task_data = {
    "absolute":
        absolute_df,

    "delta_snv":
        delta_df,
}

task_summary = pd.DataFrame([
    {
        "task":
            "absolute",

        "n_fruit_days":
            len(
                absolute_df
            ),

        "n_fruits":
            absolute_df[
                "fruit_id"
            ].nunique(),

        "days":
            "|".join(
                map(
                    str,
                    sorted(
                        absolute_df[
                            "storage_days_model"
                        ].unique()
                    )
                )
            ),

        "requires_day0_baseline":
            False,
    },
    {
        "task":
            "delta_snv",

        "n_fruit_days":
            len(
                delta_df
            ),

        "n_fruits":
            delta_df[
                "fruit_id"
            ].nunique(),

        "days":
            "|".join(
                map(
                    str,
                    sorted(
                        delta_df[
                            "storage_days_model"
                        ].unique()
                    )
                )
            ),

        "requires_day0_baseline":
            True,
    },
])

display(
    task_summary
)

task_summary.to_csv(
    TABLES /
    "02_task_summary.csv",
    index=False
)


# 5. Métricas

La selección interna usa como criterio primario:

\[
MAE_{fruit-balanced}
\]

Se reportan ocho métricas.


In [ ]:
def fruit_balanced_mae(
    df
):

    z = df.copy()

    z[
        "abs_error"
    ] = np.abs(
        z[
            "y_true"
        ]
        -
        z[
            "y_pred"
        ]
    )

    return float(
        z.groupby(
            "fruit_id"
        )[
            "abs_error"
        ]
        .mean()
        .mean()
    )


def day_balanced_mae(
    df
):

    z = df.copy()

    z[
        "abs_error"
    ] = np.abs(
        z[
            "y_true"
        ]
        -
        z[
            "y_pred"
        ]
    )

    return float(
        z.groupby(
            "y_true"
        )[
            "abs_error"
        ]
        .mean()
        .mean()
    )


def metric_bundle(
    df
):

    y = df[
        "y_true"
    ].to_numpy(
        dtype=float
    )

    p = df[
        "y_pred"
    ].to_numpy(
        dtype=float
    )

    residual = (
        p
        -
        y
    )

    rho, _ = spearmanr(
        y,
        p
    )

    return {
        "n_observations":
            int(
                len(
                    df
                )
            ),

        "n_fruits":
            int(
                df[
                    "fruit_id"
                ].nunique()
            ),

        "mae":
            float(
                mean_absolute_error(
                    y,
                    p
                )
            ),

        "rmse":
            float(
                np.sqrt(
                    mean_squared_error(
                        y,
                        p
                    )
                )
            ),

        "r2":
            float(
                r2_score(
                    y,
                    p
                )
            ),

        "spearman_rho":
            float(
                rho
            ),

        "within_1_day_accuracy":
            float(
                np.mean(
                    np.abs(
                        residual
                    )
                    <=
                    1.0
                )
            ),

        "fruit_balanced_mae":
            fruit_balanced_mae(
                df
            ),

        "day_balanced_mae":
            day_balanced_mae(
                df
            ),

        "bias":
            float(
                np.mean(
                    residual
                )
            ),
    }


# 6. Definir grillas de hiperparámetros

Las grillas son deliberadamente compactas para evitar un benchmark excesivamente exploratorio.

### Ridge
- preprocessing: RAW / SNV para tarea absoluta
- alpha: 0.01–1000

### PLSR
- componentes: 2, 5, 10, 20, 30

### SVR-RBF
- C: 1, 10, 100
- gamma: scale, 0.01
- epsilon: 0.1, 0.5

### Random Forest
- 300 árboles
- max_depth: None / 8
- min_samples_leaf: 1 / 3
- max_features: 0.5 / 1.0

### Extra Trees
- 300 árboles
- max_depth: None / 8
- min_samples_leaf: 1 / 3
- max_features: 0.5 / 1.0


In [ ]:
def config_grid(
    algorithm,
    task
):

    if task == "absolute":

        preprocessings = [
            "raw",
            "snv",
        ]

    elif task == "delta_snv":

        # La representación ya está definida como delta-SNV.
        preprocessings = [
            "as_is",
        ]

    else:

        raise ValueError(
            task
        )

    configs = []

    if algorithm == "Ridge":

        for prep, alpha in itertools.product(
            preprocessings,
            [
                0.01,
                0.1,
                1.0,
                10.0,
                100.0,
                1000.0,
            ]
        ):

            configs.append({
                "preprocessing":
                    prep,

                "alpha":
                    float(
                        alpha
                    ),
            })

    elif algorithm == "PLSR":

        for prep, n_comp in itertools.product(
            preprocessings,
            [
                2,
                5,
                10,
                20,
                30,
            ]
        ):

            configs.append({
                "preprocessing":
                    prep,

                "n_components":
                    int(
                        n_comp
                    ),
            })

    elif algorithm == "SVR_RBF":

        for (
            prep,
            C,
            gamma,
            epsilon
        ) in itertools.product(
            preprocessings,
            [
                1.0,
                10.0,
                100.0,
            ],
            [
                "scale",
                0.01,
            ],
            [
                0.1,
                0.5,
            ]
        ):

            configs.append({
                "preprocessing":
                    prep,

                "C":
                    float(
                        C
                    ),

                "gamma":
                    gamma,

                "epsilon":
                    float(
                        epsilon
                    ),
            })

    elif algorithm in [
        "RandomForest",
        "ExtraTrees",
    ]:

        for (
            prep,
            max_depth,
            min_samples_leaf,
            max_features
        ) in itertools.product(
            preprocessings,
            [
                None,
                8,
            ],
            [
                1,
                3,
            ],
            [
                0.5,
                1.0,
            ]
        ):

            configs.append({
                "preprocessing":
                    prep,

                "n_estimators":
                    300,

                "max_depth":
                    max_depth,

                "min_samples_leaf":
                    int(
                        min_samples_leaf
                    ),

                "max_features":
                    float(
                        max_features
                    ),
            })

    else:

        raise ValueError(
            algorithm
        )

    return configs


for task in TASKS:

    print("\nTASK:", task)

    for algorithm in ALGORITHMS:

        print(
            algorithm,
            len(
                config_grid(
                    algorithm,
                    task
                )
            ),
            "configs"
        )


# 7. Preprocesamiento y ajuste de algoritmos


In [ ]:
def preprocess_X(
    X,
    preprocessing
):

    X = np.asarray(
        X,
        dtype=float
    )

    if preprocessing in [
        "raw",
        "as_is",
    ]:

        return X.copy()

    if preprocessing == "snv":

        return snv_rows(
            X
        )

    raise ValueError(
        preprocessing
    )


def fit_predict_model(
    algorithm,
    config,
    X_train,
    y_train,
    X_test,
    seed
):

    preprocessing = config[
        "preprocessing"
    ]

    X_train = preprocess_X(
        X_train,
        preprocessing
    )

    X_test = preprocess_X(
        X_test,
        preprocessing
    )

    if algorithm == "Ridge":

        scaler = StandardScaler()

        Xtr = scaler.fit_transform(
            X_train
        )

        Xte = scaler.transform(
            X_test
        )

        model = Ridge(
            alpha=float(
                config[
                    "alpha"
                ]
            )
        )

        model.fit(
            Xtr,
            y_train
        )

        return np.asarray(
            model.predict(
                Xte
            ),
            dtype=float
        )

    if algorithm == "PLSR":

        scaler = StandardScaler()

        Xtr = scaler.fit_transform(
            X_train
        )

        Xte = scaler.transform(
            X_test
        )

        n_components = int(
            min(
                config[
                    "n_components"
                ],
                Xtr.shape[
                    1
                ],
                max(
                    1,
                    Xtr.shape[
                        0
                    ]
                    -
                    1
                ),
            )
        )

        model = PLSRegression(
            n_components=n_components,
            scale=False,
            max_iter=1000
        )

        model.fit(
            Xtr,
            y_train
        )

        pred = model.predict(
            Xte
        ).reshape(
            -1
        )

        return np.asarray(
            pred,
            dtype=float
        )

    if algorithm == "SVR_RBF":

        scaler = StandardScaler()

        Xtr = scaler.fit_transform(
            X_train
        )

        Xte = scaler.transform(
            X_test
        )

        model = SVR(
            kernel="rbf",
            C=float(
                config[
                    "C"
                ]
            ),
            gamma=config[
                "gamma"
            ],
            epsilon=float(
                config[
                    "epsilon"
                ]
            ),
        )

        model.fit(
            Xtr,
            y_train
        )

        return np.asarray(
            model.predict(
                Xte
            ),
            dtype=float
        )

    if algorithm == "RandomForest":

        model = RandomForestRegressor(
            n_estimators=int(
                config[
                    "n_estimators"
                ]
            ),
            max_depth=config[
                "max_depth"
            ],
            min_samples_leaf=int(
                config[
                    "min_samples_leaf"
                ]
            ),
            max_features=float(
                config[
                    "max_features"
                ]
            ),
            random_state=int(
                seed
            ),
            n_jobs=-1
        )

        model.fit(
            X_train,
            y_train
        )

        return np.asarray(
            model.predict(
                X_test
            ),
            dtype=float
        )

    if algorithm == "ExtraTrees":

        model = ExtraTreesRegressor(
            n_estimators=int(
                config[
                    "n_estimators"
                ]
            ),
            max_depth=config[
                "max_depth"
            ],
            min_samples_leaf=int(
                config[
                    "min_samples_leaf"
                ]
            ),
            max_features=float(
                config[
                    "max_features"
                ]
            ),
            random_state=int(
                seed
            ),
            n_jobs=-1
        )

        model.fit(
            X_train,
            y_train
        )

        return np.asarray(
            model.predict(
                X_test
            ),
            dtype=float
        )

    raise ValueError(
        algorithm
    )


# 8. CV interna estándar

La configuración ganadora minimiza:

1. MAE fruit-balanced
2. MAE day-balanced
3. MAE global

en ese orden.


In [ ]:
def inner_select_standard(
    task,
    algorithm,
    development_df,
    inner_assignment,
    outer_fold
):

    configs = config_grid(
        algorithm,
        task
    )

    rows = []

    best = None

    for config_idx, config in enumerate(
        configs
    ):

        pred_parts = []

        for inner_fold in range(
            INNER_FOLDS
        ):

            val_fruits = set(
                inner_assignment.loc[
                    inner_assignment[
                        "inner_fold"
                    ]
                    ==
                    inner_fold,
                    "fruit_id"
                ].astype(int)
            )

            tr = development_df[
                ~development_df[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ]

            va = development_df[
                development_df[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ]

            if len(
                va
            ) == 0:
                continue

            pred = fit_predict_model(
                algorithm=algorithm,
                config=config,
                X_train=tr[
                    wl_cols
                ].to_numpy(
                    dtype=float
                ),
                y_train=tr[
                    "storage_days_model"
                ].to_numpy(
                    dtype=float
                ),
                X_test=va[
                    wl_cols
                ].to_numpy(
                    dtype=float
                ),
                seed=(
                    RANDOM_SEED
                    +
                    outer_fold
                    *
                    1000
                    +
                    inner_fold
                    *
                    100
                    +
                    config_idx
                )
            )

            pred_parts.append(
                pd.DataFrame({
                    "fruit_id":
                        va[
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        va[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

        p = pd.concat(
            pred_parts,
            ignore_index=True
        )

        fb = fruit_balanced_mae(
            p
        )

        db = day_balanced_mae(
            p
        )

        mae = float(
            mean_absolute_error(
                p[
                    "y_true"
                ],
                p[
                    "y_pred"
                ]
            )
        )

        row = {
            "task":
                task,

            "algorithm":
                algorithm,

            "outer_fold":
                int(
                    outer_fold
                ),

            "config_idx":
                int(
                    config_idx
                ),

            "config_json":
                json.dumps(
                    config,
                    sort_keys=True
                ),

            "inner_fruit_balanced_mae":
                fb,

            "inner_day_balanced_mae":
                db,

            "inner_mae":
                mae,
        }

        rows.append(
            row
        )

        rank = (
            fb,
            db,
            mae,
            config_idx,
        )

        if (
            best is None
            or
            rank
            <
            best[
                "rank"
            ]
        ):

            best = {
                "rank":
                    rank,

                "config_idx":
                    int(
                        config_idx
                    ),

                "config":
                    config,

                "inner_fruit_balanced_mae":
                    fb,

                "inner_day_balanced_mae":
                    db,

                "inner_mae":
                    mae,
            }

    return (
        best,
        pd.DataFrame(
            rows
        )
    )


# 9. OOF estándar — 5 algoritmos × 2 tareas

Se usan checkpoints por `task × algorithm × outer fold`.


In [ ]:
standard_pred_parts = []
standard_selected_rows = []
standard_search_parts = []

for task in TASKS:

    df = task_data[
        task
    ]

    print(
        "\n",
        "=" * 72,
        sep=""
    )

    print(
        "TASK:",
        task
    )

    print(
        "=" * 72
    )

    for algorithm in ALGORITHMS:

        print(
            "\nAlgorithm:",
            algorithm
        )

        for outer_fold in range(
            OUTER_FOLDS
        ):

            stem = (
                f"{task}_"
                f"{algorithm}_"
                f"outer{outer_fold}"
            )

            pred_file = (
                CHECKPOINT_DIR /
                f"{stem}_standard_pred.csv"
            )

            selected_file = (
                CHECKPOINT_DIR /
                f"{stem}_standard_selected.json"
            )

            search_file = (
                CHECKPOINT_DIR /
                f"{stem}_standard_search.csv"
            )

            if (
                RESUME
                and
                pred_file.exists()
                and
                selected_file.exists()
                and
                search_file.exists()
            ):

                pred_df = pd.read_csv(
                    pred_file
                )

                selected_payload = json.loads(
                    selected_file.read_text(
                        encoding="utf-8"
                    )
                )

                search_df = pd.read_csv(
                    search_file
                )

                print(
                    f"  ↪ outer {outer_fold} checkpoint"
                )

            else:

                test_fruits = set(
                    outer.loc[
                        outer[
                            "outer_fold"
                        ]
                        ==
                        outer_fold,
                        "fruit_id"
                    ].astype(int)
                )

                dev = df[
                    ~df[
                        "fruit_id"
                    ].isin(
                        test_fruits
                    )
                ]

                te = df[
                    df[
                        "fruit_id"
                    ].isin(
                        test_fruits
                    )
                ]

                inner_assignment = inner[
                    inner[
                        "outer_fold"
                    ]
                    ==
                    outer_fold
                ][
                    [
                        "fruit_id",
                        "inner_fold",
                    ]
                ].copy()

                best, search_df = (
                    inner_select_standard(
                        task=task,
                        algorithm=algorithm,
                        development_df=dev,
                        inner_assignment=inner_assignment,
                        outer_fold=outer_fold
                    )
                )

                pred = fit_predict_model(
                    algorithm=algorithm,
                    config=best[
                        "config"
                    ],
                    X_train=dev[
                        wl_cols
                    ].to_numpy(
                        dtype=float
                    ),
                    y_train=dev[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),
                    X_test=te[
                        wl_cols
                    ].to_numpy(
                        dtype=float
                    ),
                    seed=(
                        RANDOM_SEED
                        +
                        outer_fold
                        *
                        1000
                        +
                        ALGORITHMS.index(
                            algorithm
                        )
                    )
                )

                pred_df = pd.DataFrame({
                    "task":
                        task,

                    "camera_type":
                        CAMERA,

                    "protocol":
                        "standard_fruit_disjoint",

                    "algorithm":
                        algorithm,

                    "outer_fold":
                        int(
                            outer_fold
                        ),

                    "config_json":
                        json.dumps(
                            best[
                                "config"
                            ],
                            sort_keys=True
                        ),

                    "fruit_id":
                        te[
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "storage_days":
                        te[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        te[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })

                selected_payload = {
                    "task":
                        task,

                    "algorithm":
                        algorithm,

                    "outer_fold":
                        int(
                            outer_fold
                        ),

                    "config":
                        best[
                            "config"
                        ],

                    "inner_fruit_balanced_mae":
                        float(
                            best[
                                "inner_fruit_balanced_mae"
                            ]
                        ),

                    "inner_day_balanced_mae":
                        float(
                            best[
                                "inner_day_balanced_mae"
                            ]
                        ),

                    "inner_mae":
                        float(
                            best[
                                "inner_mae"
                            ]
                        ),
                }

                pred_df.to_csv(
                    pred_file,
                    index=False
                )

                selected_file.write_text(
                    json.dumps(
                        selected_payload,
                        indent=2
                    ),
                    encoding="utf-8"
                )

                search_df.to_csv(
                    search_file,
                    index=False
                )

                print(
                    f"  ✓ outer {outer_fold} "
                    f"{selected_payload['config']}"
                )

            standard_pred_parts.append(
                pred_df
            )

            standard_selected_rows.append(
                {
                    "task":
                        selected_payload[
                            "task"
                        ],

                    "algorithm":
                        selected_payload[
                            "algorithm"
                        ],

                    "outer_fold":
                        int(
                            selected_payload[
                                "outer_fold"
                            ]
                        ),

                    "config_json":
                        json.dumps(
                            selected_payload[
                                "config"
                            ],
                            sort_keys=True
                        ),

                    "inner_fruit_balanced_mae":
                        float(
                            selected_payload[
                                "inner_fruit_balanced_mae"
                            ]
                        ),

                    "inner_day_balanced_mae":
                        float(
                            selected_payload[
                                "inner_day_balanced_mae"
                            ]
                        ),

                    "inner_mae":
                        float(
                            selected_payload[
                                "inner_mae"
                            ]
                        ),
                }
            )

            standard_search_parts.append(
                search_df
            )

standard_oof = pd.concat(
    standard_pred_parts,
    ignore_index=True
)

standard_selected = pd.DataFrame(
    standard_selected_rows
)

standard_search = pd.concat(
    standard_search_parts,
    ignore_index=True
)

for task in TASKS:

    expected_n = (
        199
        if task == "absolute"
        else 159
    )

    for algorithm in ALGORITHMS:

        g = standard_oof[
            (
                standard_oof[
                    "task"
                ]
                ==
                task
            )
            &
            (
                standard_oof[
                    "algorithm"
                ]
                ==
                algorithm
            )
        ]

        assert len(
            g
        ) == expected_n

print("✅ OOF estándar completo.")


# 10. Métricas estándar


In [ ]:
standard_metric_rows = []

for (
    task,
    algorithm
), g in standard_oof.groupby(
    [
        "task",
        "algorithm",
    ]
):

    m = metric_bundle(
        g
    )

    m.update({
        "task":
            task,

        "algorithm":
            algorithm,

        "protocol":
            "standard_fruit_disjoint",
    })

    standard_metric_rows.append(
        m
    )

standard_metrics = pd.DataFrame(
    standard_metric_rows
).sort_values(
    [
        "task",
        "fruit_balanced_mae",
    ]
)

display(
    standard_metrics
)

standard_metrics.to_csv(
    TABLES /
    "03_standard_algorithm_metrics.csv",
    index=False
)

standard_selected.to_csv(
    TABLES /
    "04_standard_selected_configs.csv",
    index=False
)

standard_search.to_csv(
    TABLES /
    "05_standard_inner_search.csv",
    index=False
)


# 11. Null estándar para cada tarea


In [ ]:
standard_null_parts = []

for task in TASKS:

    df = task_data[
        task
    ]

    for outer_fold in range(
        OUTER_FOLDS
    ):

        test_fruits = set(
            outer.loc[
                outer[
                    "outer_fold"
                ]
                ==
                outer_fold,
                "fruit_id"
            ].astype(int)
        )

        tr = df[
            ~df[
                "fruit_id"
            ].isin(
                test_fruits
            )
        ]

        te = df[
            df[
                "fruit_id"
            ].isin(
                test_fruits
            )
        ]

        null_value = float(
            np.median(
                tr[
                    "storage_days_model"
                ]
            )
        )

        standard_null_parts.append(
            pd.DataFrame({
                "task":
                    task,

                "protocol":
                    "standard_null",

                "algorithm":
                    "DummyMedian",

                "outer_fold":
                    int(
                        outer_fold
                    ),

                "fruit_id":
                    te[
                        "fruit_id"
                    ].to_numpy(
                        dtype=int
                    ),

                "storage_days":
                    te[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=int
                    ),

                "y_true":
                    te[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),

                "y_pred":
                    np.repeat(
                        null_value,
                        len(
                            te
                        )
                    ),
            })
        )

standard_null_oof = pd.concat(
    standard_null_parts,
    ignore_index=True
)

standard_null_metric_rows = []

for task, g in standard_null_oof.groupby(
    "task"
):

    m = metric_bundle(
        g
    )

    m.update({
        "task":
            task,

        "algorithm":
            "DummyMedian",

        "protocol":
            "standard_null",
    })

    standard_null_metric_rows.append(
        m
    )

standard_null_metrics = pd.DataFrame(
    standard_null_metric_rows
)

display(
    standard_null_metrics
)


# 12. CV interna session-excluded

Para cada target day \(d\):

- se elimina \(d\) del training;
- la configuración se selecciona con inner CV bajo la misma regla;
- el test externo conserva solo el día \(d\).

El null correcto también excluye \(d\).


In [ ]:
def inner_select_session_excluded(
    task,
    algorithm,
    development_df,
    inner_assignment,
    outer_fold
):

    configs = config_grid(
        algorithm,
        task
    )

    rows = []

    best = None

    for config_idx, config in enumerate(
        configs
    ):

        pred_parts = []

        for inner_fold in range(
            INNER_FOLDS
        ):

            val_fruits = set(
                inner_assignment.loc[
                    inner_assignment[
                        "inner_fold"
                    ]
                    ==
                    inner_fold,
                    "fruit_id"
                ].astype(int)
            )

            tr_all = development_df[
                ~development_df[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ]

            va_all = development_df[
                development_df[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ]

            if len(
                va_all
            ) == 0:
                continue

            for target_day in sorted(
                va_all[
                    "storage_days_model"
                ].unique()
            ):

                tr = tr_all[
                    tr_all[
                        "storage_days_model"
                    ]
                    !=
                    target_day
                ]

                va = va_all[
                    va_all[
                        "storage_days_model"
                    ]
                    ==
                    target_day
                ]

                if len(
                    va
                ) == 0:
                    continue

                pred = fit_predict_model(
                    algorithm=algorithm,
                    config=config,
                    X_train=tr[
                        wl_cols
                    ].to_numpy(
                        dtype=float
                    ),
                    y_train=tr[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),
                    X_test=va[
                        wl_cols
                    ].to_numpy(
                        dtype=float
                    ),
                    seed=(
                        RANDOM_SEED
                        +
                        outer_fold
                        *
                        10000
                        +
                        inner_fold
                        *
                        1000
                        +
                        int(
                            target_day
                        )
                        *
                        10
                        +
                        config_idx
                    )
                )

                pred_parts.append(
                    pd.DataFrame({
                        "fruit_id":
                            va[
                                "fruit_id"
                            ].to_numpy(
                                dtype=int
                            ),

                        "y_true":
                            va[
                                "storage_days_model"
                            ].to_numpy(
                                dtype=float
                            ),

                        "y_pred":
                            pred,
                    })
                )

        p = pd.concat(
            pred_parts,
            ignore_index=True
        )

        fb = fruit_balanced_mae(
            p
        )

        db = day_balanced_mae(
            p
        )

        mae = float(
            mean_absolute_error(
                p[
                    "y_true"
                ],
                p[
                    "y_pred"
                ]
            )
        )

        row = {
            "task":
                task,

            "algorithm":
                algorithm,

            "outer_fold":
                int(
                    outer_fold
                ),

            "config_idx":
                int(
                    config_idx
                ),

            "config_json":
                json.dumps(
                    config,
                    sort_keys=True
                ),

            "inner_fruit_balanced_mae":
                fb,

            "inner_day_balanced_mae":
                db,

            "inner_mae":
                mae,
        }

        rows.append(
            row
        )

        rank = (
            fb,
            db,
            mae,
            config_idx,
        )

        if (
            best is None
            or
            rank
            <
            best[
                "rank"
            ]
        ):

            best = {
                "rank":
                    rank,

                "config_idx":
                    int(
                        config_idx
                    ),

                "config":
                    config,

                "inner_fruit_balanced_mae":
                    fb,

                "inner_day_balanced_mae":
                    db,

                "inner_mae":
                    mae,
            }

    return (
        best,
        pd.DataFrame(
            rows
        )
    )


# 13. OOF session-excluded — 5 algoritmos × 2 tareas

Esta sección es la más lenta del notebook.  
Los checkpoints permiten reanudar si Colab se interrumpe.


In [ ]:
session_pred_parts = []
session_selected_rows = []
session_search_parts = []

for task in TASKS:

    df = task_data[
        task
    ]

    print(
        "\n",
        "=" * 72,
        sep=""
    )

    print(
        "SESSION-EXCLUDED TASK:",
        task
    )

    print(
        "=" * 72
    )

    for algorithm in ALGORITHMS:

        print(
            "\nAlgorithm:",
            algorithm
        )

        for outer_fold in range(
            OUTER_FOLDS
        ):

            stem = (
                f"{task}_"
                f"{algorithm}_"
                f"outer{outer_fold}"
            )

            pred_file = (
                CHECKPOINT_DIR /
                f"{stem}_session_pred.csv"
            )

            selected_file = (
                CHECKPOINT_DIR /
                f"{stem}_session_selected.json"
            )

            search_file = (
                CHECKPOINT_DIR /
                f"{stem}_session_search.csv"
            )

            if (
                RESUME
                and
                pred_file.exists()
                and
                selected_file.exists()
                and
                search_file.exists()
            ):

                pred_df = pd.read_csv(
                    pred_file
                )

                selected_payload = json.loads(
                    selected_file.read_text(
                        encoding="utf-8"
                    )
                )

                search_df = pd.read_csv(
                    search_file
                )

                print(
                    f"  ↪ outer {outer_fold} checkpoint"
                )

            else:

                test_fruits = set(
                    outer.loc[
                        outer[
                            "outer_fold"
                        ]
                        ==
                        outer_fold,
                        "fruit_id"
                    ].astype(int)
                )

                dev = df[
                    ~df[
                        "fruit_id"
                    ].isin(
                        test_fruits
                    )
                ]

                te_all = df[
                    df[
                        "fruit_id"
                    ].isin(
                        test_fruits
                    )
                ]

                inner_assignment = inner[
                    inner[
                        "outer_fold"
                    ]
                    ==
                    outer_fold
                ][
                    [
                        "fruit_id",
                        "inner_fold",
                    ]
                ].copy()

                best, search_df = (
                    inner_select_session_excluded(
                        task=task,
                        algorithm=algorithm,
                        development_df=dev,
                        inner_assignment=inner_assignment,
                        outer_fold=outer_fold
                    )
                )

                fold_pred_parts = []

                for target_day in sorted(
                    te_all[
                        "storage_days_model"
                    ].unique()
                ):

                    tr = dev[
                        dev[
                            "storage_days_model"
                        ]
                        !=
                        target_day
                    ]

                    te = te_all[
                        te_all[
                            "storage_days_model"
                        ]
                        ==
                        target_day
                    ]

                    pred = fit_predict_model(
                        algorithm=algorithm,
                        config=best[
                            "config"
                        ],
                        X_train=tr[
                            wl_cols
                        ].to_numpy(
                            dtype=float
                        ),
                        y_train=tr[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),
                        X_test=te[
                            wl_cols
                        ].to_numpy(
                            dtype=float
                        ),
                        seed=(
                            RANDOM_SEED
                            +
                            outer_fold
                            *
                            10000
                            +
                            int(
                                target_day
                            )
                            *
                            100
                            +
                            ALGORITHMS.index(
                                algorithm
                            )
                        )
                    )

                    fold_pred_parts.append(
                        pd.DataFrame({
                            "task":
                                task,

                            "camera_type":
                                CAMERA,

                            "protocol":
                                "session_excluded",

                            "algorithm":
                                algorithm,

                            "outer_fold":
                                int(
                                    outer_fold
                                ),

                            "target_day":
                                int(
                                    target_day
                                ),

                            "config_json":
                                json.dumps(
                                    best[
                                        "config"
                                    ],
                                    sort_keys=True
                                ),

                            "fruit_id":
                                te[
                                    "fruit_id"
                                ].to_numpy(
                                    dtype=int
                                ),

                            "storage_days":
                                te[
                                    "storage_days_model"
                                ].to_numpy(
                                    dtype=int
                                ),

                            "y_true":
                                te[
                                    "storage_days_model"
                                ].to_numpy(
                                    dtype=float
                                ),

                            "y_pred":
                                pred,
                        })
                    )

                pred_df = pd.concat(
                    fold_pred_parts,
                    ignore_index=True
                )

                selected_payload = {
                    "task":
                        task,

                    "algorithm":
                        algorithm,

                    "outer_fold":
                        int(
                            outer_fold
                        ),

                    "config":
                        best[
                            "config"
                        ],

                    "inner_fruit_balanced_mae":
                        float(
                            best[
                                "inner_fruit_balanced_mae"
                            ]
                        ),

                    "inner_day_balanced_mae":
                        float(
                            best[
                                "inner_day_balanced_mae"
                            ]
                        ),

                    "inner_mae":
                        float(
                            best[
                                "inner_mae"
                            ]
                        ),
                }

                pred_df.to_csv(
                    pred_file,
                    index=False
                )

                selected_file.write_text(
                    json.dumps(
                        selected_payload,
                        indent=2
                    ),
                    encoding="utf-8"
                )

                search_df.to_csv(
                    search_file,
                    index=False
                )

                print(
                    f"  ✓ outer {outer_fold} "
                    f"{selected_payload['config']}"
                )

            session_pred_parts.append(
                pred_df
            )

            session_selected_rows.append(
                {
                    "task":
                        selected_payload[
                            "task"
                        ],

                    "algorithm":
                        selected_payload[
                            "algorithm"
                        ],

                    "outer_fold":
                        int(
                            selected_payload[
                                "outer_fold"
                            ]
                        ),

                    "config_json":
                        json.dumps(
                            selected_payload[
                                "config"
                            ],
                            sort_keys=True
                        ),

                    "inner_fruit_balanced_mae":
                        float(
                            selected_payload[
                                "inner_fruit_balanced_mae"
                            ]
                        ),

                    "inner_day_balanced_mae":
                        float(
                            selected_payload[
                                "inner_day_balanced_mae"
                            ]
                        ),

                    "inner_mae":
                        float(
                            selected_payload[
                                "inner_mae"
                            ]
                        ),
                }
            )

            session_search_parts.append(
                search_df
            )

session_oof = pd.concat(
    session_pred_parts,
    ignore_index=True
)

session_selected = pd.DataFrame(
    session_selected_rows
)

session_search = pd.concat(
    session_search_parts,
    ignore_index=True
)

for task in TASKS:

    expected_n = (
        199
        if task == "absolute"
        else 159
    )

    for algorithm in ALGORITHMS:

        g = session_oof[
            (
                session_oof[
                    "task"
                ]
                ==
                task
            )
            &
            (
                session_oof[
                    "algorithm"
                ]
                ==
                algorithm
            )
        ]

        assert len(
            g
        ) == expected_n

print("✅ OOF session-excluded completo.")


# 14. Métricas session-excluded


In [ ]:
session_metric_rows = []

for (
    task,
    algorithm
), g in session_oof.groupby(
    [
        "task",
        "algorithm",
    ]
):

    m = metric_bundle(
        g
    )

    m.update({
        "task":
            task,

        "algorithm":
            algorithm,

        "protocol":
            "session_excluded",
    })

    session_metric_rows.append(
        m
    )

session_metrics = pd.DataFrame(
    session_metric_rows
).sort_values(
    [
        "task",
        "fruit_balanced_mae",
    ]
)

display(
    session_metrics
)

session_metrics.to_csv(
    TABLES /
    "06_session_excluded_algorithm_metrics.csv",
    index=False
)

session_selected.to_csv(
    TABLES /
    "07_session_excluded_selected_configs.csv",
    index=False
)

session_search.to_csv(
    TABLES /
    "08_session_excluded_inner_search.csv",
    index=False
)


# 15. Null correcto session-excluded para cada tarea


In [ ]:
session_null_parts = []

for task in TASKS:

    df = task_data[
        task
    ]

    for outer_fold in range(
        OUTER_FOLDS
    ):

        test_fruits = set(
            outer.loc[
                outer[
                    "outer_fold"
                ]
                ==
                outer_fold,
                "fruit_id"
            ].astype(int)
        )

        dev = df[
            ~df[
                "fruit_id"
            ].isin(
                test_fruits
            )
        ]

        te_all = df[
            df[
                "fruit_id"
            ].isin(
                test_fruits
            )
        ]

        for target_day in sorted(
            te_all[
                "storage_days_model"
            ].unique()
        ):

            tr = dev[
                dev[
                    "storage_days_model"
                ]
                !=
                target_day
            ]

            te = te_all[
                te_all[
                    "storage_days_model"
                ]
                ==
                target_day
            ]

            null_value = float(
                tr[
                    "storage_days_model"
                ].mean()
            )

            session_null_parts.append(
                pd.DataFrame({
                    "task":
                        task,

                    "protocol":
                        "session_excluded_null",

                    "algorithm":
                        "SessionExcluded_DummyMean",

                    "outer_fold":
                        int(
                            outer_fold
                        ),

                    "target_day":
                        int(
                            target_day
                        ),

                    "fruit_id":
                        te[
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "storage_days":
                        te[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        te[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        np.repeat(
                            null_value,
                            len(
                                te
                            )
                        ),
                })
            )

session_null_oof = pd.concat(
    session_null_parts,
    ignore_index=True
)

session_null_metric_rows = []

for task, g in session_null_oof.groupby(
    "task"
):

    m = metric_bundle(
        g
    )

    m.update({
        "task":
            task,

        "algorithm":
            "SessionExcluded_DummyMean",

        "protocol":
            "session_excluded_null",
    })

    session_null_metric_rows.append(
        m
    )

session_null_metrics = pd.DataFrame(
    session_null_metric_rows
)

display(
    session_null_metrics
)


# 16. Ranking consolidado

Se crean rankings por:

- MAE fruit-balanced
- MAE
- RMSE
- \(R^2\)
- Spearman
- ±1 día

El ranking principal del paper debe mantenerse basado en MAE fruit-balanced.


In [ ]:
def rank_metrics(
    df
):

    z = df.copy()

    z[
        "rank_fruit_balanced_mae"
    ] = z[
        "fruit_balanced_mae"
    ].rank(
        method="min",
        ascending=True
    )

    z[
        "rank_mae"
    ] = z[
        "mae"
    ].rank(
        method="min",
        ascending=True
    )

    z[
        "rank_rmse"
    ] = z[
        "rmse"
    ].rank(
        method="min",
        ascending=True
    )

    z[
        "rank_r2"
    ] = z[
        "r2"
    ].rank(
        method="min",
        ascending=False
    )

    z[
        "rank_spearman"
    ] = z[
        "spearman_rho"
    ].rank(
        method="min",
        ascending=False
    )

    z[
        "rank_within_1_day"
    ] = z[
        "within_1_day_accuracy"
    ].rank(
        method="min",
        ascending=False
    )

    z[
        "mean_rank_secondary"
    ] = z[
        [
            "rank_mae",
            "rank_rmse",
            "rank_r2",
            "rank_spearman",
            "rank_within_1_day",
        ]
    ].mean(
        axis=1
    )

    return z


ranking_parts = []

for protocol_name, metrics_df in [
    (
        "standard",
        standard_metrics
    ),
    (
        "session_excluded",
        session_metrics
    ),
]:

    for task, g in metrics_df.groupby(
        "task"
    ):

        r = rank_metrics(
            g
        )

        r[
            "ranking_protocol"
        ] = protocol_name

        ranking_parts.append(
            r
        )

ranking = pd.concat(
    ranking_parts,
    ignore_index=True
)

display(
    ranking.sort_values(
        [
            "ranking_protocol",
            "task",
            "rank_fruit_balanced_mae",
        ]
    )
)

ranking.to_csv(
    TABLES /
    "09_algorithm_rankings.csv",
    index=False
)


# 17. Bootstrap pareado por fruto

Se compara cada algoritmo contra Ridge dentro de la misma tarea y protocolo.

Además, el mejor algoritmo por MAE fruit-balanced se compara contra el null correspondiente.

Estas comparaciones son exploratorias/post-selección.


In [ ]:
def per_fruit_mae(
    df
):

    z = df.copy()

    z[
        "abs_error"
    ] = np.abs(
        z[
            "y_true"
        ]
        -
        z[
            "y_pred"
        ]
    )

    return (
        z.groupby(
            "fruit_id"
        )[
            "abs_error"
        ]
        .mean()
    )


def paired_bootstrap(
    a,
    b,
    label,
    n_boot=5000,
    seed=20260818
):

    ids = sorted(
        set(
            a.index
        )
        &
        set(
            b.index
        )
    )

    av = a.loc[
        ids
    ].to_numpy(
        dtype=float
    )

    bv = b.loc[
        ids
    ].to_numpy(
        dtype=float
    )

    diff = (
        av
        -
        bv
    )

    rng = np.random.default_rng(
        seed
    )

    boot = np.empty(
        n_boot,
        dtype=float
    )

    for i in range(
        n_boot
    ):

        idx = rng.integers(
            0,
            len(
                diff
            ),
            size=len(
                diff
            )
        )

        boot[
            i
        ] = np.mean(
            diff[
                idx
            ]
        )

    return {
        "comparison":
            label,

        "n_fruits":
            int(
                len(
                    diff
                )
            ),

        "mean_delta_mae_days":
            float(
                diff.mean()
            ),

        "ci95_low":
            float(
                np.percentile(
                    boot,
                    2.5
                )
            ),

        "ci95_high":
            float(
                np.percentile(
                    boot,
                    97.5
                )
            ),

        "fraction_first_better":
            float(
                np.mean(
                    diff
                    <
                    0
                )
            ),
    }


bootstrap_rows = []
seed_offset = 0

for protocol_name, pred_df, metrics_df, null_df in [
    (
        "standard",
        standard_oof,
        standard_metrics,
        standard_null_oof
    ),
    (
        "session_excluded",
        session_oof,
        session_metrics,
        session_null_oof
    ),
]:

    for task in TASKS:

        ridge = pred_df[
            (
                pred_df[
                    "task"
                ]
                ==
                task
            )
            &
            (
                pred_df[
                    "algorithm"
                ]
                ==
                "Ridge"
            )
        ]

        ridge_pf = per_fruit_mae(
            ridge
        )

        for algorithm in ALGORITHMS:

            if algorithm == "Ridge":
                continue

            g = pred_df[
                (
                    pred_df[
                        "task"
                    ]
                    ==
                    task
                )
                &
                (
                    pred_df[
                        "algorithm"
                    ]
                    ==
                    algorithm
                )
            ]

            bootstrap_rows.append(
                paired_bootstrap(
                    per_fruit_mae(
                        g
                    ),
                    ridge_pf,
                    (
                        f"{protocol_name} {task} "
                        f"{algorithm} minus Ridge"
                    ),
                    BOOTSTRAP_REPS,
                    RANDOM_SEED + seed_offset
                )
            )

            seed_offset += 1

        best_algorithm = (
            metrics_df[
                metrics_df[
                    "task"
                ]
                ==
                task
            ]
            .sort_values(
                "fruit_balanced_mae"
            )
            .iloc[
                0
            ][
                "algorithm"
            ]
        )

        best_pred = pred_df[
            (
                pred_df[
                    "task"
                ]
                ==
                task
            )
            &
            (
                pred_df[
                    "algorithm"
                ]
                ==
                best_algorithm
            )
        ]

        null_task = null_df[
            null_df[
                "task"
            ]
            ==
            task
        ]

        bootstrap_rows.append(
            paired_bootstrap(
                per_fruit_mae(
                    best_pred
                ),
                per_fruit_mae(
                    null_task
                ),
                (
                    f"{protocol_name} {task} "
                    f"best({best_algorithm}) minus null"
                ),
                BOOTSTRAP_REPS,
                RANDOM_SEED + seed_offset
            )
        )

        seed_offset += 1

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

display(
    bootstrap_results
)

bootstrap_results.to_csv(
    TABLES /
    "10_paired_fruit_bootstrap_algorithms.csv",
    index=False
)


# 18. Métricas por día


In [ ]:
day_rows = []

for protocol_name, pred_df in [
    (
        "standard",
        standard_oof
    ),
    (
        "session_excluded",
        session_oof
    ),
]:

    for (
        task,
        algorithm,
        day
    ), g in pred_df.groupby(
        [
            "task",
            "algorithm",
            "storage_days",
        ]
    ):

        err = (
            g[
                "y_pred"
            ].to_numpy(
                dtype=float
            )
            -
            g[
                "y_true"
            ].to_numpy(
                dtype=float
            )
        )

        day_rows.append({
            "protocol":
                protocol_name,

            "task":
                task,

            "algorithm":
                algorithm,

            "storage_days":
                int(
                    day
                ),

            "n":
                int(
                    len(
                        g
                    )
                ),

            "mae":
                float(
                    np.mean(
                        np.abs(
                            err
                        )
                    )
                ),

            "rmse":
                float(
                    np.sqrt(
                        np.mean(
                            err ** 2
                        )
                    )
                ),

            "bias":
                float(
                    np.mean(
                        err
                    )
                ),
        })

day_metrics = pd.DataFrame(
    day_rows
)

display(
    day_metrics
)

day_metrics.to_csv(
    TABLES /
    "11_metrics_by_day.csv",
    index=False
)


# 19. Estabilidad de configuraciones seleccionadas


In [ ]:
config_stability_standard = (
    standard_selected.groupby(
        [
            "task",
            "algorithm",
            "config_json",
        ]
    )
    .size()
    .reset_index(
        name="n_outer_folds_selected"
    )
    .sort_values(
        [
            "task",
            "algorithm",
            "n_outer_folds_selected",
        ],
        ascending=[
            True,
            True,
            False,
        ]
    )
)

config_stability_session = (
    session_selected.groupby(
        [
            "task",
            "algorithm",
            "config_json",
        ]
    )
    .size()
    .reset_index(
        name="n_outer_folds_selected"
    )
    .sort_values(
        [
            "task",
            "algorithm",
            "n_outer_folds_selected",
        ],
        ascending=[
            True,
            True,
            False,
        ]
    )
)

display(
    config_stability_standard
)

config_stability_standard.to_csv(
    TABLES /
    "12_standard_config_stability.csv",
    index=False
)

config_stability_session.to_csv(
    TABLES /
    "13_session_config_stability.csv",
    index=False
)


# 20. Figura — benchmark estándar


In [ ]:
for task in TASKS:

    g = standard_metrics[
        standard_metrics[
            "task"
        ]
        ==
        task
    ].sort_values(
        "fruit_balanced_mae"
    )

    null_mae = float(
        standard_null_metrics.loc[
            standard_null_metrics[
                "task"
            ]
            ==
            task,
            "fruit_balanced_mae"
        ].iloc[
            0
        ]
    )

    fig, ax = plt.subplots(
        figsize=(9, 5)
    )

    ax.bar(
        g[
            "algorithm"
        ],
        g[
            "fruit_balanced_mae"
        ]
    )

    ax.axhline(
        null_mae,
        linestyle="--",
        label="DummyMedian"
    )

    ax.set_ylabel(
        "MAE fruit-balanced (días)"
    )

    ax.set_title(
        f"VIS_COR — benchmark estándar — {task}"
    )

    ax.tick_params(
        axis="x",
        rotation=15
    )

    ax.legend()

    ax.grid(
        axis="y",
        alpha=0.2
    )

    fig.tight_layout()

    fig.savefig(
        FIGURES /
        f"01_standard_benchmark_{task}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# 21. Figura — benchmark session-excluded


In [ ]:
for task in TASKS:

    g = session_metrics[
        session_metrics[
            "task"
        ]
        ==
        task
    ].sort_values(
        "fruit_balanced_mae"
    )

    null_mae = float(
        session_null_metrics.loc[
            session_null_metrics[
                "task"
            ]
            ==
            task,
            "fruit_balanced_mae"
        ].iloc[
            0
        ]
    )

    fig, ax = plt.subplots(
        figsize=(9, 5)
    )

    ax.bar(
        g[
            "algorithm"
        ],
        g[
            "fruit_balanced_mae"
        ]
    )

    ax.axhline(
        null_mae,
        linestyle="--",
        label="SessionExcluded_DummyMean"
    )

    ax.set_ylabel(
        "MAE fruit-balanced (días)"
    )

    ax.set_title(
        f"VIS_COR — session-excluded — {task}"
    )

    ax.tick_params(
        axis="x",
        rotation=15
    )

    ax.legend()

    ax.grid(
        axis="y",
        alpha=0.2
    )

    fig.tight_layout()

    fig.savefig(
        FIGURES /
        f"02_session_excluded_benchmark_{task}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# 22. Figura — comparación de las métricas principales del mejor algoritmo

Para cada tarea se identifica el algoritmo con menor MAE fruit-balanced estándar.


In [ ]:
best_rows = []

for task in TASKS:

    best = (
        standard_metrics[
            standard_metrics[
                "task"
            ]
            ==
            task
        ]
        .sort_values(
            "fruit_balanced_mae"
        )
        .iloc[
            0
        ]
    )

    best_rows.append(
        best
    )

best_standard = pd.DataFrame(
    best_rows
)

display(
    best_standard
)

best_standard.to_csv(
    TABLES /
    "14_best_standard_algorithm_by_task.csv",
    index=False
)


# 23. Tabla maestra para manuscrito


In [ ]:
master_metrics = pd.concat(
    [
        standard_metrics,
        standard_null_metrics,
        session_metrics,
        session_null_metrics,
    ],
    ignore_index=True,
    sort=False
)

columns_order = [
    "protocol",
    "task",
    "algorithm",
    "n_observations",
    "n_fruits",
    "mae",
    "rmse",
    "r2",
    "spearman_rho",
    "within_1_day_accuracy",
    "fruit_balanced_mae",
    "day_balanced_mae",
    "bias",
]

master_metrics = master_metrics[
    [
        c
        for c in columns_order
        if c in master_metrics.columns
    ]
]

display(
    master_metrics
)

master_metrics.to_csv(
    TABLES /
    "15_master_algorithm_metrics_for_manuscript.csv",
    index=False
)


# 24. Guardar resultados centrales


In [ ]:
STANDARD_OOF_PATH = (
    RESULTS /
    "kaki_final_benchmark_standard_oof.csv"
)

SESSION_OOF_PATH_OUT = (
    RESULTS /
    "kaki_final_benchmark_session_excluded_oof.csv"
)

STANDARD_NULL_PATH = (
    RESULTS /
    "kaki_final_benchmark_standard_null_oof.csv"
)

SESSION_NULL_PATH = (
    RESULTS /
    "kaki_final_benchmark_session_null_oof.csv"
)

MASTER_METRICS_PATH = (
    RESULTS /
    "kaki_final_benchmark_master_metrics.csv"
)

standard_oof.to_csv(
    STANDARD_OOF_PATH,
    index=False
)

session_oof.to_csv(
    SESSION_OOF_PATH_OUT,
    index=False
)

standard_null_oof.to_csv(
    STANDARD_NULL_PATH,
    index=False
)

session_null_oof.to_csv(
    SESSION_NULL_PATH,
    index=False
)

master_metrics.to_csv(
    MASTER_METRICS_PATH,
    index=False
)

print("✅ Resultados guardados.")


# 25. Resumen científico automático


In [ ]:
def json_safe(
    obj
):

    if isinstance(
        obj,
        dict
    ):
        return {
            str(k):
                json_safe(
                    v
                )
            for k, v in obj.items()
        }

    if isinstance(
        obj,
        (
            list,
            tuple,
        )
    ):
        return [
            json_safe(
                v
            )
            for v in obj
        ]

    if isinstance(
        obj,
        np.integer
    ):
        return int(
            obj
        )

    if isinstance(
        obj,
        np.floating
    ):
        return (
            None
            if np.isnan(
                obj
            )
            else float(
                obj
            )
        )

    if isinstance(
        obj,
        np.bool_
    ):
        return bool(
            obj
        )

    if isinstance(
        obj,
        np.ndarray
    ):
        return json_safe(
            obj.tolist()
        )

    return obj


best_standard_payload = []

for task in TASKS:

    row = (
        standard_metrics[
            standard_metrics[
                "task"
            ]
            ==
            task
        ]
        .sort_values(
            "fruit_balanced_mae"
        )
        .iloc[
            0
        ]
    )

    best_standard_payload.append(
        row.to_dict()
    )


best_session_payload = []

for task in TASKS:

    row = (
        session_metrics[
            session_metrics[
                "task"
            ]
            ==
            task
        ]
        .sort_values(
            "fruit_balanced_mae"
        )
        .iloc[
            0
        ]
    )

    best_session_payload.append(
        row.to_dict()
    )


summary = {
    "project":
        PROJECT_NAME,

    "notebook":
        NOTEBOOK_ID,

    "run_signature":
        RUN_SIGNATURE,

    "timestamp":
        datetime.now().isoformat(
            timespec="seconds"
        ),

    "camera":
        CAMERA,

    "algorithms":
        ALGORITHMS,

    "n_algorithms":
        len(
            ALGORITHMS
        ),

    "reported_metrics": [
        "MAE",
        "RMSE",
        "R2",
        "Spearman rho",
        "within ±1 day accuracy",
        "fruit-balanced MAE",
        "day-balanced MAE",
        "bias",
    ],

    "primary_metric":
        "fruit-balanced MAE",

    "tasks": {
        "absolute": {
            "n_fruit_days":
                199,

            "n_fruits":
                40,

            "requires_day0_baseline":
                False,
        },

        "delta_snv": {
            "n_fruit_days":
                159,

            "n_fruits":
                38,

            "requires_day0_baseline":
                True,
        },
    },

    "standard_metrics":
        standard_metrics.to_dict(
            orient="records"
        ),

    "session_excluded_metrics":
        session_metrics.to_dict(
            orient="records"
        ),

    "standard_null_metrics":
        standard_null_metrics.to_dict(
            orient="records"
        ),

    "session_excluded_null_metrics":
        session_null_metrics.to_dict(
            orient="records"
        ),

    "best_standard_by_task":
        best_standard_payload,

    "best_session_excluded_by_task":
        best_session_payload,

    "paired_bootstrap":
        bootstrap_results.to_dict(
            orient="records"
        ),

    "limitations": [
        (
            "Algorithm comparisons are internal to one acquisition campaign."
        ),
        (
            "Hyperparameters are selected by nested fruit-disjoint CV, but "
            "post-hoc paired bootstrap comparisons remain exploratory."
        ),
        (
            "The delta-SNV task requires a baseline measurement from the same fruit."
        ),
        (
            "Session-excluded remains a stress test and not a causal removal "
            "of day/session confounding."
        ),
        (
            "External campaign validation remains necessary before deployment claims."
        ),
    ],
}

SUMMARY_PATH = (
    RESULTS /
    "kaki_final_classical_ml_benchmark_summary.json"
)

SUMMARY_PATH.write_text(
    json.dumps(
        json_safe(
            summary
        ),
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        json_safe(
            summary
        ),
        indent=2,
        ensure_ascii=False
    )
)


# 26. Notas automáticas de revisión


In [ ]:
review_lines = [
    "KAKI — FINAL CLASSICAL ML BENCHMARK",
    "",
    f"Camera: {CAMERA}",
    f"Algorithms: {', '.join(ALGORITHMS)}",
    "",
    "Primary metric: fruit-balanced MAE",
    "",
    "STANDARD:",
]

for task in TASKS:

    review_lines.append(
        f"  TASK = {task}"
    )

    g = standard_metrics[
        standard_metrics[
            "task"
        ]
        ==
        task
    ].sort_values(
        "fruit_balanced_mae"
    )

    for _, r in g.iterrows():

        review_lines.append(
            (
                f"    {r['algorithm']}: "
                f"FB-MAE={r['fruit_balanced_mae']:.4f}, "
                f"MAE={r['mae']:.4f}, "
                f"RMSE={r['rmse']:.4f}, "
                f"R2={r['r2']:.4f}, "
                f"rho={r['spearman_rho']:.4f}, "
                f"±1d={r['within_1_day_accuracy']:.4f}, "
                f"day-bal={r['day_balanced_mae']:.4f}, "
                f"bias={r['bias']:.4f}"
            )
        )

    review_lines.append(
        ""
    )

review_lines.append(
    "SESSION-EXCLUDED:"
)

for task in TASKS:

    review_lines.append(
        f"  TASK = {task}"
    )

    g = session_metrics[
        session_metrics[
            "task"
        ]
        ==
        task
    ].sort_values(
        "fruit_balanced_mae"
    )

    for _, r in g.iterrows():

        review_lines.append(
            (
                f"    {r['algorithm']}: "
                f"FB-MAE={r['fruit_balanced_mae']:.4f}, "
                f"MAE={r['mae']:.4f}, "
                f"RMSE={r['rmse']:.4f}, "
                f"R2={r['r2']:.4f}, "
                f"rho={r['spearman_rho']:.4f}, "
                f"±1d={r['within_1_day_accuracy']:.4f}"
            )
        )

    null_value = float(
        session_null_metrics.loc[
            session_null_metrics[
                "task"
            ]
            ==
            task,
            "fruit_balanced_mae"
        ].iloc[
            0
        ]
    )

    review_lines.append(
        (
            f"    Correct null FB-MAE={null_value:.4f}"
        )
    )

    review_lines.append(
        ""
    )

review_lines.extend([
    "Interpretation:",
    (
        "This benchmark tests whether the main conclusions are robust to "
        "multiple linear and nonlinear regression families."
    ),
    (
        "Do not select the paper's scientific conclusion solely from the "
        "lowest MAE. Preserve the validation hierarchy and the distinction "
        "between single-scan absolute prediction and longitudinal delta-SNV tracking."
    ),
    (
        "After reviewing this package, the main classical-model phase can be frozen "
        "and the manuscript tables/figures can be consolidated."
    ),
])

REVIEW_PATH = (
    RESULTS /
    "FINAL_CLASSICAL_BENCHMARK_REVIEW_NOTES.txt"
)

REVIEW_PATH.write_text(
    "\n".join(
        review_lines
    )
    +
    "\n",
    encoding="utf-8"
)

print(
    REVIEW_PATH.read_text()
)
